# Finn FTMO Live Trading System ($25,000 Account)

**Objectives:**
- Profit Target: $2,500 (10%)
- Max Daily Loss: $1250 (5%)
- Max Total Loss: $2,500 (10%)
- Min Trading Days: 4 (Monday to Thursday)

**Strategy Portfolio:**
1. Moving Average Crossover (Short-term momentum)
2. Momentum Breakout (Range breakouts)
3. Mean Reversion RSI (Oversold/overbought)
4. Trend Following ATR (Strong trends)
5. Volume Profile Breakout (High-probability zones)

## System Configuration

In [ ]:
import sys
print(f"Python version: {sys.version}")

## MT5 Account Configuration

In [ ]:
# ============================================================================
# FTMO MT5 ACCOUNT CREDENTIALS
# ============================================================================

USE_REAL_MT5_ACCOUNT = True

# REAL LIVE ACCOUNT
MT5_ACCOUNT = "********"
MT5_PASSWORD = "********"
MT5_SERVER = "FTMO-Server-Real"

# Account Size
ACCOUNT_SIZE = 25000.00

# DEMO ACCOUNT
# MT5_ACCOUNT = "********"
# MT5_PASSWORD = "********"
# MT5_SERVER = "FTMO-Demo"

# Account Size
# ACCOUNT_SIZE = 100000.00

# ============================================================================
# FTMO Server Options:
# - FTMO-Demo         (Free Trial)
# - FTMO-Server       (Challenge/Verification)
# - FTMO-Server2      (Challenge/Verification)
# - FTMO-Server3      (Challenge/Verification)
# - FTMO-Server4      (Challenge/Verification)
# - FTMO-Server-Real  (Funded)
# ============================================================================

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import MetaTrader5 as mt5
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple, Union
import time
from dataclasses import dataclass, field
import logging
from decimal import Decimal, ROUND_HALF_UP
from tabulate import tabulate
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

## FTMO Parameters for $25,000 Account

In [ ]:
@dataclass
class FTMOParameters:
    """FTMO challenge parameters for $25,000 account."""
    
    account_size: float = 25000.00
    
    # FTMO Objectives
    profit_target: float = 25000.00          # 10% of account 2500
    max_daily_loss: float = 1250.00          # 5% of account 1250
    max_total_loss: float = 25000.00          # 10% of account 2500
    min_trading_days: int = 4
    
    # Risk Management
    max_risk_per_trade: float = 0.015       # 1.5% per trade
    max_portfolio_risk: float = 0.03        # 3% total across all positions
    max_concurrent_trades: int = 3
    
    # Position Sizing Constraints
    min_stop_loss_pips: float = 10.0
    max_stop_loss_pips: float = 50.0
    min_lot_size: float = 0.01
    max_lot_size: float = 2.00
    
    # Strategy Allocation (risk per strategy)
    strategy_risk_weights: Dict[str, float] = field(default_factory=lambda: {
        'MA_Crossover': 0.25,
        'Momentum_Breakout': 0.20,
        'Mean_Reversion': 0.20,
        'Trend_Following': 0.20,
        'Volume_Breakout': 0.15
    })
    
    def get_daily_remaining_loss(self, current_daily_pnl: float) -> float:
        """Calculate remaining allowable loss for the day."""
        return self.max_daily_loss + current_daily_pnl
    
    def get_total_remaining_loss(self, total_pnl: float) -> float:
        """Calculate remaining allowable total loss."""
        return self.max_total_loss + total_pnl
    
    def can_trade(self, current_daily_pnl: float, total_pnl: float) -> Tuple[bool, str]:
        """Check if trading is allowed based on FTMO limits."""
        if current_daily_pnl <= -self.max_daily_loss:
            return False, "Daily loss limit reached"
        if total_pnl <= -self.max_total_loss:
            return False, "Total loss limit reached"
        if total_pnl >= self.profit_target:
            return False, "Profit target achieved"
        return True, "Trading allowed"

ftmo_params = FTMOParameters()
print(f"FTMO Parameters Initialised:")
print(f"  Account Size: ${ftmo_params.account_size:,.2f}")
print(f"  Profit Target: ${ftmo_params.profit_target:,.2f}")
print(f"  Max Daily Loss: ${ftmo_params.max_daily_loss:,.2f}")
print(f"  Max Total Loss: ${ftmo_params.max_total_loss:,.2f}")
print(f"  Min Trading Days: {ftmo_params.min_trading_days}")
print(f"  Max Risk/Trade: {ftmo_params.max_risk_per_trade*100:.1f}%")

## Forex Pair Specifications

In [ ]:
@dataclass
class ForexPairSpecification:
    """Forex pair specifications for accurate calculations."""
    
    symbol: str
    pip_value: float
    point_value: float
    digits: int
    contract_size: float = 25000.0
    min_lot: float = 0.01
    max_lot: float = 100.0
    lot_step: float = 0.01
    typical_spread_pips: float = 1.0
    
    def calculate_pip_value_for_lot(self, lot_size: float) -> float:
        """Calculate pip value for specific lot size."""
        return self.pip_value * lot_size
    
    def pips_to_price(self, pips: float) -> float:
        """Convert pips to price movement."""
        multiplier = 10 if self.digits == 5 or self.digits == 3 else 1
        return pips * self.point_value * multiplier
    
    def price_to_pips(self, price_diff: float) -> float:
        """Convert price difference to pips."""
        multiplier = 10 if self.digits == 5 or self.digits == 3 else 1
        return abs(price_diff) / (self.point_value * multiplier)

FOREX_SPECS = {
    'EURUSD': ForexPairSpecification('EURUSD', 10.0, 0.00001, 5, typical_spread_pips=0.8),
    'GBPUSD': ForexPairSpecification('GBPUSD', 10.0, 0.00001, 5, typical_spread_pips=1.2),
    'USDJPY': ForexPairSpecification('USDJPY', 9.16, 0.001, 3, typical_spread_pips=0.9),
    'AUDUSD': ForexPairSpecification('AUDUSD', 10.0, 0.00001, 5, typical_spread_pips=1.0),
    'USDCAD': ForexPairSpecification('USDCAD', 7.87, 0.00001, 5, typical_spread_pips=1.2),
    'USDCHF': ForexPairSpecification('USDCHF', 11.17, 0.00001, 5, typical_spread_pips=1.5),
    'NZDUSD': ForexPairSpecification('NZDUSD', 10.0, 0.00001, 5, typical_spread_pips=1.5),
    'EURGBP': ForexPairSpecification('EURGBP', 12.74, 0.00001, 5, typical_spread_pips=1.0),
    'EURJPY': ForexPairSpecification('EURJPY', 9.16, 0.001, 3, typical_spread_pips=1.2),
    'GBPJPY': ForexPairSpecification('GBPJPY', 9.16, 0.001, 3, typical_spread_pips=1.8)
}

print("Forex Specifications Loaded:")
for symbol, spec in FOREX_SPECS.items():
    print(f"  {symbol}: Pip=${spec.pip_value}/lot, Spread={spec.typical_spread_pips} pips")

## Position Size Calculator

In [ ]:
class PositionSizeCalculator:
    """Calculate position sizes based on FTMO risk parameters."""
    
    def __init__(self, ftmo_params: FTMOParameters):
        self.ftmo_params = ftmo_params
        self.logger = logging.getLogger(self.__class__.__name__)
    
    def calculate_position_size(
        self,
        account_balance: float,
        stop_loss_pips: float,
        pair_spec: ForexPairSpecification,
        risk_percentage: float
    ) -> Dict[str, float]:
        """
        Calculate position size based on account balance and stop loss.
        
        Args:
            account_balance: Current account balance
            stop_loss_pips: Stop loss distance in pips
            pair_spec: Forex pair specification
            risk_percentage: Risk as decimal (e.g., 0.01 for 1%)
        
        Returns:
            Dictionary with lot size and risk amount
        """
        risk_amount = account_balance * risk_percentage
        
        # Ensure stop loss within limits
        stop_loss_pips = max(
            self.ftmo_params.min_stop_loss_pips,
            min(stop_loss_pips, self.ftmo_params.max_stop_loss_pips)
        )
        
        # Calculate lot size
        pip_value_per_lot = pair_spec.pip_value
        lot_size = risk_amount / (stop_loss_pips * pip_value_per_lot)
        
        # Round to lot step
        lot_size = round(lot_size / pair_spec.lot_step) * pair_spec.lot_step
        
        # Apply limits
        lot_size = max(
            self.ftmo_params.min_lot_size,
            min(lot_size, self.ftmo_params.max_lot_size)
        )
        
        actual_risk = lot_size * pip_value_per_lot * stop_loss_pips
        
        return {
            'lot_size': lot_size,
            'risk_amount': actual_risk,
            'risk_percentage': (actual_risk / account_balance) * 100,
            'stop_loss_pips': stop_loss_pips
        }
    
    def calculate_take_profit(
        self,
        entry_price: float,
        stop_loss_pips: float,
        reward_risk_ratio: float,
        pair_spec: ForexPairSpecification,
        direction: str
    ) -> Dict[str, float]:
        """
        Calculate take profit based on reward:risk ratio.
        
        Args:
            entry_price: Entry price
            stop_loss_pips: Stop loss in pips
            reward_risk_ratio: Reward to risk ratio
            pair_spec: Forex pair specification
            direction: 'BUY' or 'SELL'
        
        Returns:
            Dictionary with take profit price and pips
        """
        tp_pips = stop_loss_pips * reward_risk_ratio
        tp_price_change = pair_spec.pips_to_price(tp_pips)
        
        if direction.upper() == 'BUY':
            tp_price = entry_price + tp_price_change
        else:
            tp_price = entry_price - tp_price_change
        
        return {
            'take_profit_price': tp_price,
            'take_profit_pips': tp_pips
        }

position_calc = PositionSizeCalculator(ftmo_params)
print("Position Size Calculator Initialised")

## Market Data Handler

In [ ]:
class MarketDataHandler:
    """Handle market data retrieval from MT5."""
    
    def __init__(self, symbol: str, timeframe: int):
        self.symbol = symbol
        self.timeframe = timeframe
        self.logger = logging.getLogger(self.__class__.__name__)
    
    def get_historical_data(self, bars: int = 500) -> pd.DataFrame:
        """Retrieve historical data from MT5."""
        rates = mt5.copy_rates_from_pos(self.symbol, self.timeframe, 0, bars)
        
        if rates is None:
            self.logger.error(f"Failed to retrieve data for {self.symbol}")
            return pd.DataFrame()
        
        df = pd.DataFrame(rates)
        df['time'] = pd.to_datetime(df['time'], unit='s')
        
        return df
    
    def get_current_price(self) -> Dict[str, float]:
        """Get current bid/ask prices."""
        tick = mt5.symbol_info_tick(self.symbol)
        
        if tick is None:
            self.logger.error(f"Failed to get tick for {self.symbol}")
            return {'bid': 0.0, 'ask': 0.0}
        
        return {
            'bid': tick.bid,
            'ask': tick.ask,
            'time': datetime.fromtimestamp(tick.time)
        }

print("Market Data Handler Initialised")

## Trading Strategies

In [ ]:
class BaseStrategy:
    """Base class for trading strategies."""
    
    def __init__(self, name: str, risk_per_trade: float, reward_risk_ratio: float):
        self.name = name
        self.risk_per_trade = risk_per_trade
        self.reward_risk_ratio = reward_risk_ratio
        self.logger = logging.getLogger(name)
    
    def calculate_signals(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate trading signals. Override in subclass."""
        raise NotImplementedError
    
    def get_current_signal(self, df: pd.DataFrame) -> int:
        """Get current signal: 1=BUY, -1=SELL, 0=NEUTRAL."""
        if 'signal' not in df.columns or len(df) == 0:
            return 0
        return int(df['signal'].iloc[-1])


class MACrossoverStrategy(BaseStrategy):
    """Moving Average Crossover strategy."""
    
    def __init__(self):
        super().__init__(
            name='MA Crossover',
            risk_per_trade=0.012,
            reward_risk_ratio=2.5
        )
        self.fast_period = 9
        self.slow_period = 21
    
    def calculate_signals(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate MA crossover signals."""
        df = df.copy()
        
        df['ma_fast'] = df['close'].rolling(window=self.fast_period).mean()
        df['ma_slow'] = df['close'].rolling(window=self.slow_period).mean()
        
        df['signal'] = 0
        df.loc[df['ma_fast'] > df['ma_slow'], 'signal'] = 1
        df.loc[df['ma_fast'] < df['ma_slow'], 'signal'] = -1
        
        # ATR for stop loss
        df['atr'] = self._calculate_atr(df)
        
        return df
    
    def _calculate_atr(self, df: pd.DataFrame, period: int = 14) -> pd.Series:
        """Calculate Average True Range."""
        high_low = df['high'] - df['low']
        high_close = np.abs(df['high'] - df['close'].shift())
        low_close = np.abs(df['low'] - df['close'].shift())
        
        tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        return tr.rolling(window=period).mean()


class MomentumBreakoutStrategy(BaseStrategy):
    """Momentum Breakout strategy."""
    
    def __init__(self):
        super().__init__(
            name='Momentum Breakout',
            risk_per_trade=0.010,
            reward_risk_ratio=3.0
        )
        self.lookback_period = 20
    
    def calculate_signals(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate momentum breakout signals."""
        df = df.copy()
        
        df['high_level'] = df['high'].rolling(window=self.lookback_period).max()
        df['low_level'] = df['low'].rolling(window=self.lookback_period).min()
        
        df['signal'] = 0
        df.loc[df['close'] > df['high_level'].shift(1), 'signal'] = 1
        df.loc[df['close'] < df['low_level'].shift(1), 'signal'] = -1
        
        df['atr'] = self._calculate_atr(df)
        
        return df
    
    def _calculate_atr(self, df: pd.DataFrame, period: int = 14) -> pd.Series:
        """Calculate Average True Range."""
        high_low = df['high'] - df['low']
        high_close = np.abs(df['high'] - df['close'].shift())
        low_close = np.abs(df['low'] - df['close'].shift())
        
        tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        return tr.rolling(window=period).mean()


class MeanReversionRSIStrategy(BaseStrategy):
    """Mean Reversion RSI strategy."""
    
    def __init__(self):
        super().__init__(
            name='Mean Reversion RSI',
            risk_per_trade=0.008,
            reward_risk_ratio=2.0
        )
        self.rsi_period = 14
        self.oversold = 30
        self.overbought = 70
    
    def calculate_signals(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate RSI mean reversion signals."""
        df = df.copy()
        
        df['rsi'] = self._calculate_rsi(df['close'], self.rsi_period)
        
        df['signal'] = 0
        df.loc[df['rsi'] < self.oversold, 'signal'] = 1
        df.loc[df['rsi'] > self.overbought, 'signal'] = -1
        
        df['atr'] = self._calculate_atr(df)
        
        return df
    
    def _calculate_rsi(self, prices: pd.Series, period: int) -> pd.Series:
        """Calculate RSI."""
        delta = prices.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        
        rs = gain / loss
        return 100 - (100 / (1 + rs))
    
    def _calculate_atr(self, df: pd.DataFrame, period: int = 14) -> pd.Series:
        """Calculate Average True Range."""
        high_low = df['high'] - df['low']
        high_close = np.abs(df['high'] - df['close'].shift())
        low_close = np.abs(df['low'] - df['close'].shift())
        
        tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        return tr.rolling(window=period).mean()


class TrendFollowingATRStrategy(BaseStrategy):
    """Trend Following with ATR strategy."""
    
    def __init__(self):
        super().__init__(
            name='Trend Following ATR',
            risk_per_trade=0.012,
            reward_risk_ratio=3.5
        )
        self.ma_period = 50
        self.atr_period = 14
    
    def calculate_signals(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate trend following signals."""
        df = df.copy()
        
        df['ma'] = df['close'].rolling(window=self.ma_period).mean()
        df['atr'] = self._calculate_atr(df, self.atr_period)
        
        df['signal'] = 0
        df.loc[df['close'] > df['ma'], 'signal'] = 1
        df.loc[df['close'] < df['ma'], 'signal'] = -1
        
        return df
    
    def _calculate_atr(self, df: pd.DataFrame, period: int) -> pd.Series:
        """Calculate Average True Range."""
        high_low = df['high'] - df['low']
        high_close = np.abs(df['high'] - df['close'].shift())
        low_close = np.abs(df['low'] - df['close'].shift())
        
        tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        return tr.rolling(window=period).mean()


class VolumeProfileBreakoutStrategy(BaseStrategy):
    """Volume Profile Breakout strategy."""
    
    def __init__(self):
        super().__init__(
            name='Volume Profile Breakout',
            risk_per_trade=0.010,
            reward_risk_ratio=2.5
        )
        self.volume_ma_period = 20
        self.breakout_period = 10
    
    def calculate_signals(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate volume breakout signals."""
        df = df.copy()
        
        df['volume_ma'] = df['tick_volume'].rolling(window=self.volume_ma_period).mean()
        df['high_level'] = df['high'].rolling(window=self.breakout_period).max()
        df['low_level'] = df['low'].rolling(window=self.breakout_period).min()
        
        df['signal'] = 0
        
        # Buy signal: price breaks high with volume confirmation
        buy_condition = (
            (df['close'] > df['high_level'].shift(1)) &
            (df['tick_volume'] > df['volume_ma'])
        )
        df.loc[buy_condition, 'signal'] = 1
        
        # Sell signal: price breaks low with volume confirmation
        sell_condition = (
            (df['close'] < df['low_level'].shift(1)) &
            (df['tick_volume'] > df['volume_ma'])
        )
        df.loc[sell_condition, 'signal'] = -1
        
        df['atr'] = self._calculate_atr(df)
        
        return df
    
    def _calculate_atr(self, df: pd.DataFrame, period: int = 14) -> pd.Series:
        """Calculate Average True Range."""
        high_low = df['high'] - df['low']
        high_close = np.abs(df['high'] - df['close'].shift())
        low_close = np.abs(df['low'] - df['close'].shift())
        
        tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        return tr.rolling(window=period).mean()

print("Trading Strategies Loaded")

## Real-Time Risk Monitor

In [ ]:
class RiskMonitor:
    """Monitor FTMO risk limits in real-time."""
    
    def __init__(self, ftmo_params: FTMOParameters):
        self.ftmo_params = ftmo_params
        self.trading_days = set()
        self.daily_pnl = 0.0
        self.total_pnl = 0.0
        self.current_date = None
        self.logger = logging.getLogger(self.__class__.__name__)
    
    def update_from_positions(self):
        """Update PnL from MT5 positions and history."""
        # Get today's date
        today = datetime.now().date()
        
        # Reset daily PnL if new day
        if self.current_date != today:
            self.current_date = today
            self.daily_pnl = 0.0
        
        # Get closed positions from today
        from_date = datetime.combine(today, datetime.min.time())
        to_date = datetime.now()
        
        deals = mt5.history_deals_get(from_date, to_date)
        if deals is not None and len(deals) > 0:
            deals_df = pd.DataFrame(list(deals), columns=deals[0]._asdict().keys())
            # Filter only closed positions (type 1)
            closed_deals = deals_df[deals_df['entry'] == 1]
            if len(closed_deals) > 0:
                self.daily_pnl = closed_deals['profit'].sum()
                if len(closed_deals) > 0:
                    self.trading_days.add(today)
        
        # Get open positions
        positions = mt5.positions_get()
        open_pnl = 0.0
        if positions is not None and len(positions) > 0:
            open_pnl = sum(pos.profit for pos in positions)
        
        # Total daily includes closed + open
        self.daily_pnl = self.daily_pnl + open_pnl
        
        # Calculate total PnL
        account = mt5.account_info()
        if account:
            self.total_pnl = account.equity - self.ftmo_params.account_size
    
    def check_limits(self) -> Tuple[bool, List[str]]:
        """
        Check if FTMO limits are respected.
        
        Returns:
            Tuple of (can_trade, warnings)
        """
        self.update_from_positions()
        
        warnings = []
        can_trade = True
        
        # Check daily loss
        daily_loss = abs(min(0, self.daily_pnl))
        if daily_loss >= self.ftmo_params.max_daily_loss:
            warnings.append(f"BREACH: Daily loss ${daily_loss:.2f} >= ${self.ftmo_params.max_daily_loss:.2f}")
            can_trade = False
        elif daily_loss >= self.ftmo_params.max_daily_loss * 0.8:
            warnings.append(f"WARNING: Daily loss ${daily_loss:.2f} approaching limit")
        
        # Check total loss
        total_loss = abs(min(0, self.total_pnl))
        if total_loss >= self.ftmo_params.max_total_loss:
            warnings.append(f"BREACH: Total loss ${total_loss:.2f} >= ${self.ftmo_params.max_total_loss:.2f}")
            can_trade = False
        elif total_loss >= self.ftmo_params.max_total_loss * 0.8:
            warnings.append(f"WARNING: Total loss ${total_loss:.2f} approaching limit")
        
        # Check profit target
        if self.total_pnl >= self.ftmo_params.profit_target:
            warnings.append(f"TARGET ACHIEVED: Profit ${self.total_pnl:.2f}")
            can_trade = False
        
        return can_trade, warnings
    
    def get_status(self) -> Dict:
        """Get current risk status."""
        self.update_from_positions()
        
        return {
            'daily_pnl': self.daily_pnl,
            'total_pnl': self.total_pnl,
            'daily_loss_remaining': self.ftmo_params.max_daily_loss + min(0, self.daily_pnl),
            'total_loss_remaining': self.ftmo_params.max_total_loss + min(0, self.total_pnl),
            'profit_to_target': self.ftmo_params.profit_target - self.total_pnl,
            'trading_days': len(self.trading_days),
            'days_needed': max(0, self.ftmo_params.min_trading_days - len(self.trading_days))
        }

risk_monitor = RiskMonitor(ftmo_params)
print("Risk Monitor Initialised")

## Trade Execution Engine

In [ ]:
class TradeExecutor:
    """Execute trades on MT5."""
    
    def __init__(self, magic_number: int = 234000):
        self.magic_number = magic_number
        self.logger = logging.getLogger(self.__class__.__name__)
    
    def execute_trade(
        self,
        symbol: str,
        direction: str,
        lot_size: float,
        entry_price: float,
        stop_loss: float,
        take_profit: float,
        comment: str = ""
    ) -> Dict:
        """
        Execute trade order.
        
        Args:
            symbol: Trading symbol
            direction: 'BUY' or 'SELL'
            lot_size: Position size in lots
            entry_price: Entry price
            stop_loss: Stop loss price
            take_profit: Take profit price
            comment: Order comment
        
        Returns:
            Dictionary with order result
        """
        # Determine order type
        if direction.upper() == 'BUY':
            order_type = mt5.ORDER_TYPE_BUY
            price = mt5.symbol_info_tick(symbol).ask
        else:
            order_type = mt5.ORDER_TYPE_SELL
            price = mt5.symbol_info_tick(symbol).bid
        
        # Prepare request
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot_size,
            "type": order_type,
            "price": price,
            "sl": stop_loss,
            "tp": take_profit,
            "deviation": 20,
            "magic": self.magic_number,
            "comment": comment,
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_IOC,
        }
        
        # Send order
        result = mt5.order_send(request)
        
        if result.retcode != mt5.TRADE_RETCODE_DONE:
            self.logger.error(f"Order failed: {result.comment}")
            return {
                'success': False,
                'error': result.comment,
                'retcode': result.retcode
            }
        
        self.logger.info(f"Order executed: {direction} {lot_size} lots {symbol} @ {price}")
        
        return {
            'success': True,
            'order': result.order,
            'volume': result.volume,
            'price': result.price,
            'comment': result.comment
        }
    
    def close_all_positions(self, symbol: Optional[str] = None):
        """Close all open positions."""
        positions = mt5.positions_get(symbol=symbol) if symbol else mt5.positions_get()
        
        if positions is None or len(positions) == 0:
            self.logger.info("No positions to close")
            return
        
        for position in positions:
            tick = mt5.symbol_info_tick(position.symbol)
            
            request = {
                "action": mt5.TRADE_ACTION_DEAL,
                "symbol": position.symbol,
                "volume": position.volume,
                "type": mt5.ORDER_TYPE_SELL if position.type == 0 else mt5.ORDER_TYPE_BUY,
                "position": position.ticket,
                "price": tick.bid if position.type == 0 else tick.ask,
                "deviation": 20,
                "magic": self.magic_number,
                "comment": "Close position",
                "type_time": mt5.ORDER_TIME_GTC,
                "type_filling": mt5.ORDER_FILLING_IOC,
            }
            
            result = mt5.order_send(request)
            if result.retcode == mt5.TRADE_RETCODE_DONE:
                self.logger.info(f"Closed position {position.ticket}")
            else:
                self.logger.error(f"Failed to close {position.ticket}: {result.comment}")

trade_executor = TradeExecutor()
print("Trade Executor Initialised")

## Initialise MT5 Connection

In [ ]:
# Initialise MT5
if not mt5.initialize():
    print(f"❌ MT5 initialisation failed: {mt5.last_error()}")
else:
    print("✓ MT5 initialised")
    
    # Login if credentials provided
    if USE_REAL_MT5_ACCOUNT:
        authorized = mt5.login(
            login=MT5_ACCOUNT,
            password=MT5_PASSWORD,
            server=MT5_SERVER
        )
        
        if not authorized:
            print(f"❌ MT5 login failed: {mt5.last_error()}")
        else:
            account = mt5.account_info()
            print("\n" + "="*80)
            print("MT5 ACCOUNT CONNECTED")
            print("="*80)
            print(f"Account: {account.login}")
            print(f"Server: {account.server}")
            print(f"Balance: ${account.balance:,.2f}")
            print(f"Equity: ${account.equity:,.2f}")
            print(f"Margin: ${account.margin:,.2f}")
            print(f"Free Margin: ${account.margin_free:,.2f}")
            print(f"Leverage: 1:{account.leverage}")
            print("="*80)

## Live Trading System

In [ ]:
# Configuration
SYMBOL = 'EURUSD'
TIMEFRAME = mt5.TIMEFRAME_M5
pair_spec = FOREX_SPECS[SYMBOL]

print("\n" + "="*80)
print("LIVE TRADING SYSTEM INITIALISATION")
print("="*80)

# Initialise components             
market_data = MarketDataHandler(SYMBOL, TIMEFRAME)

strategies = [
    MACrossoverStrategy(),
    MomentumBreakoutStrategy(),
    MeanReversionRSIStrategy(),
    TrendFollowingATRStrategy(),
    VolumeProfileBreakoutStrategy()
]

print(f"\n✓ Trading Pair: {SYMBOL}")
print(f"  Timeframe: M5")
print(f"  Strategies: {len(strategies)}")
print(f"\n✓ FTMO Parameters:")
print(f"  Account: ${ftmo_params.account_size:,.2f}")
print(f"  Profit Target: ${ftmo_params.profit_target:,.2f}")
print(f"  Max Daily Loss: ${ftmo_params.max_daily_loss:,.2f}")
print(f"  Max Total Loss: ${ftmo_params.max_total_loss:,.2f}")
print(f"  Min Trading Days: {ftmo_params.min_trading_days}")

# Get current market data
current_price = market_data.get_current_price()
df = market_data.get_historical_data(bars=200)

print(f"\n✓ Current Market:")
print(f"  Bid: {current_price['bid']:.{pair_spec.digits}f}")
print(f"  Ask: {current_price['ask']:.{pair_spec.digits}f}")
print(f"  Spread: {pair_spec.price_to_pips(current_price['ask'] - current_price['bid']):.1f} pips")
print(f"\n✓ Historical Data: {len(df)} bars loaded")

print("\n" + "="*80)
print("SYSTEM READY")
print("="*80)

## Generate Trading Signals

In [ ]:
# Check risk limits first
can_trade, warnings = risk_monitor.check_limits()
risk_status = risk_monitor.get_status()

print("\n" + "="*80)
print("RISK STATUS")
print("="*80)
print(f"Daily P&L: ${risk_status['daily_pnl']:,.2f}")
print(f"Total P&L: ${risk_status['total_pnl']:,.2f}")
print(f"Daily Loss Remaining: ${risk_status['daily_loss_remaining']:,.2f}")
print(f"Total Loss Remaining: ${risk_status['total_loss_remaining']:,.2f}")
print(f"Profit to Target: ${risk_status['profit_to_target']:,.2f}")
print(f"Trading Days: {risk_status['trading_days']}/{ftmo_params.min_trading_days}")
print(f"Days Needed: {risk_status['days_needed']}")

if warnings:
    print("\nWARNINGS:")
    for warning in warnings:
        print(f"  ⚠ {warning}")

print(f"\nTrading Allowed: {'✓ YES' if can_trade else '✗ NO'}")

if not can_trade:
    print("\n⛔ Trading halted due to risk limits")
else:
    print("\n" + "="*80)
    print("STRATEGY SIGNALS")
    print("="*80)
    
    signal_results = []
    account = mt5.account_info()
    
    for strategy in strategies:
        df_signals = strategy.calculate_signals(df)
        current_signal = strategy.get_current_signal(df_signals)
        signal_text = "BUY" if current_signal == 1 else "SELL" if current_signal == -1 else "NEUTRAL"
        
        if current_signal != 0:
            # Calculate stop loss from ATR
            if 'atr' in df_signals.columns:
                atr_value = df_signals['atr'].iloc[-1]
                stop_loss_pips = pair_spec.price_to_pips(atr_value * 1.5)
            else:
                stop_loss_pips = 20.0
            
            stop_loss_pips = max(
                ftmo_params.min_stop_loss_pips,
                min(stop_loss_pips, ftmo_params.max_stop_loss_pips)
            )
            
            # Calculate position size
            position_info = position_calc.calculate_position_size(
                account.balance,
                stop_loss_pips,
                pair_spec,
                strategy.risk_per_trade
            )
            
            # Entry and SL prices
            if current_signal == 1:
                entry_price = current_price['ask']
                sl_price = entry_price - pair_spec.pips_to_price(stop_loss_pips)
            else:
                entry_price = current_price['bid']
                sl_price = entry_price + pair_spec.pips_to_price(stop_loss_pips)
            
            # Calculate TP
            tp_info = position_calc.calculate_take_profit(
                entry_price,
                stop_loss_pips,
                strategy.reward_risk_ratio,
                pair_spec,
                signal_text
            )
            
            # Calculate profit/loss amounts
            pip_value = pair_spec.calculate_pip_value_for_lot(position_info['lot_size'])
            potential_loss = stop_loss_pips * pip_value
            potential_profit = tp_info['take_profit_pips'] * pip_value
            
            signal_results.append({
                'Strategy': strategy.name,
                'Signal': signal_text,
                'Entry': f"{entry_price:.{pair_spec.digits}f}",
                'SL': f"{sl_price:.{pair_spec.digits}f}",
                'TP': f"{tp_info['take_profit_price']:.{pair_spec.digits}f}",
                'SL Pips': f"{stop_loss_pips:.1f}",
                'TP Pips': f"{tp_info['take_profit_pips']:.1f}",
                'Lot Size': f"{position_info['lot_size']:.2f}",
                'Risk $': f"${potential_loss:.2f}",
                'Profit $': f"${potential_profit:.2f}",
                'R:R': f"{strategy.reward_risk_ratio}:1"
            })
        else:
            signal_results.append({
                'Strategy': strategy.name,
                'Signal': signal_text,
                'Entry': '-', 'SL': '-', 'TP': '-',
                'SL Pips': '-', 'TP Pips': '-', 'Lot Size': '-',
                'Risk $': '-', 'Profit $': '-',
                'R:R': f"{strategy.reward_risk_ratio}:1"
            })
    
    signals_df = pd.DataFrame(signal_results)
    print("\n" + tabulate(signals_df, headers='keys', tablefmt='grid', showindex=False))
    
    active_signals = len([s for s in signal_results if s['Signal'] != 'NEUTRAL'])
    print(f"\nActive Signals: {active_signals}")
    print(f"Max Concurrent Trades: {ftmo_params.max_concurrent_trades}")

## Execute Trades (Manual Trigger)

In [ ]:
# MANUAL EXECUTION - Run this cell to execute trades based on signals
# WARNING: This will place real trades on your MT5 account

EXECUTE_TRADES = True  # Set to True to execute

if not EXECUTE_TRADES:
    print("⚠ Trade execution disabled. Set EXECUTE_TRADES = True to execute.")
else:
    can_trade, warnings = risk_monitor.check_limits()
    
    if not can_trade:
        print("⛔ Cannot execute trades - risk limits reached")
    else:
        # Get current positions
        current_positions = mt5.positions_get()
        position_count = len(current_positions) if current_positions else 0
        
        if position_count >= ftmo_params.max_concurrent_trades:
            print(f"⛔ Max concurrent trades ({ftmo_params.max_concurrent_trades}) reached")
        else:
            print("\n" + "="*80)
            print("EXECUTING TRADES")
            print("="*80)
            
            executed_count = 0
            
            for idx, signal in enumerate(signal_results):
                if signal['Signal'] != 'NEUTRAL' and executed_count < ftmo_params.max_concurrent_trades - position_count:
                    strategy = strategies[idx]
                    
                    # Parse values
                    entry = float(signal['Entry'])
                    sl = float(signal['SL'])
                    tp = float(signal['TP'])
                    lot_size = float(signal['Lot Size'])
                    
                    result = trade_executor.execute_trade(
                        symbol=SYMBOL,
                        direction=signal['Signal'],
                        lot_size=lot_size,
                        entry_price=entry,
                        stop_loss=sl,
                        take_profit=tp,
                        comment=f"{strategy.name}"
                    )
                    
                    if result['success']:
                        print(f"\n✓ Trade Executed: {strategy.name}")
                        print(f"  Direction: {signal['Signal']}")
                        print(f"  Lot Size: {lot_size}")
                        print(f"  Entry: {entry}")
                        print(f"  SL: {sl} ({signal['SL Pips']} pips)")
                        print(f"  TP: {tp} ({signal['TP Pips']} pips)")
                        print(f"  Risk: {signal['Risk $']}")
                        print(f"  Potential Profit: {signal['Profit $']}")
                        executed_count += 1
                    else:
                        print(f"\n✗ Trade Failed: {strategy.name}")
                        print(f"  Error: {result.get('error', 'Unknown')}")
            
            print(f"\n" + "="*80)
            print(f"Trades Executed: {executed_count}")
            print("="*80)

## Monitor Active Positions

In [ ]:
# Get open positions
positions = mt5.positions_get()

print("\n" + "="*80)
print("ACTIVE POSITIONS")
print("="*80)

if positions is None or len(positions) == 0:
    print("No active positions")
else:
    position_data = []
    
    for pos in positions:
        direction = "BUY" if pos.type == 0 else "SELL"
        pips = pair_spec.price_to_pips(pos.price_current - pos.price_open)
        if pos.type == 1:  # SELL
            pips = -pips
        
        position_data.append({
            'Ticket': pos.ticket,
            'Symbol': pos.symbol,
            'Type': direction,
            'Lots': pos.volume,
            'Open': f"{pos.price_open:.{pair_spec.digits}f}",
            'Current': f"{pos.price_current:.{pair_spec.digits}f}",
            'SL': f"{pos.sl:.{pair_spec.digits}f}" if pos.sl > 0 else '-',
            'TP': f"{pos.tp:.{pair_spec.digits}f}" if pos.tp > 0 else '-',
            'Pips': f"{pips:.1f}",
            'Profit': f"${pos.profit:.2f}",
            'Comment': pos.comment
        })
    
    positions_df = pd.DataFrame(position_data)
    print("\n" + tabulate(positions_df, headers='keys', tablefmt='grid', showindex=False))
    
    total_profit = sum(pos.profit for pos in positions)
    print(f"\nTotal Open P&L: ${total_profit:.2f}")
    print(f"Position Count: {len(positions)}/{ftmo_params.max_concurrent_trades}")

## Close All Positions (Emergency)

In [ ]:
# EMERGENCY CLOSE - Run this to close all positions
# WARNING: This will close all open positions immediately

CLOSE_ALL = True  # Set to True to close all positions

if not CLOSE_ALL:
    print("⚠ Position closure disabled. Set CLOSE_ALL = True to close all positions.")
else:
    print("\n⚠ CLOSING ALL POSITIONS...")
    trade_executor.close_all_positions()
    print("✓ All positions closed")

## Daily Summary

In [ ]:
# Get account info
account = mt5.account_info()
risk_status = risk_monitor.get_status()

print("\n" + "="*80)
print("DAILY SUMMARY")
print("="*80)

print(f"\nAccount Information:")
print(f"  Balance: ${account.balance:,.2f}")
print(f"  Equity: ${account.equity:,.2f}")
print(f"  Margin Used: ${account.margin:,.2f}")
print(f"  Free Margin: ${account.margin_free:,.2f}")

print(f"\nFTMO Progress:")
print(f"  Daily P&L: ${risk_status['daily_pnl']:,.2f}")
print(f"  Total P&L: ${risk_status['total_pnl']:,.2f}")
print(f"  Profit to Target: ${risk_status['profit_to_target']:,.2f}")
print(f"  Target Progress: {(risk_status['total_pnl']/ftmo_params.profit_target)*100:.1f}%")

print(f"\nRisk Limits:")
print(f"  Daily Loss Used: ${abs(min(0, risk_status['daily_pnl'])):,.2f} / ${ftmo_params.max_daily_loss:,.2f}")
print(f"  Total Loss Used: ${abs(min(0, risk_status['total_pnl'])):,.2f} / ${ftmo_params.max_total_loss:,.2f}")

print(f"\nTrading Days:")
print(f"  Completed: {risk_status['trading_days']}")
print(f"  Required: {ftmo_params.min_trading_days}")
print(f"  Remaining: {risk_status['days_needed']}")

# Get positions
positions = mt5.positions_get()
position_count = len(positions) if positions else 0

print(f"\nPositions:")
print(f"  Active: {position_count}")
print(f"  Max Allowed: {ftmo_params.max_concurrent_trades}")

if positions:
    total_open_pnl = sum(pos.profit for pos in positions)
    print(f"  Open P&L: ${total_open_pnl:.2f}")

print("\n" + "="*80)

## Shutdown MT5 Connection

In [ ]:
# Shutdown MT5 when done
mt5.shutdown()
print("✓ MT5 connection closed")